# Experimental Study

### Loading Libraries

In [2]:
%cd ../..

/Users/joaquinromero/Desktop


In [3]:
# Installing darts library to recreate experiments
!pip install darts

  Using cached pytorch_lightning-2.5.2-py3-none-any.whl.metadata (21 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.2 MB/s eta 0:00:00
Using cached pytorch_lightning-2.5.2-py3-none-any.whl (825 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.6/547.6 kB 23.7 MB/s eta 0:00:00
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
  Attempting uninstall: pytorch-lightning
    Found existing installation: pytorch-lightning 2.5.4
    Uninstalling pytorch-lightning-2.5.4:
      Successfully uninstalled pytorch-lightning-2.5.4


In [ ]:
# Numerical Computing
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Notebook Optimizer
from tqdm.autonotebook import tqdm

# OS
import os
import random
import warnings
from pathlib import Path

# DARTS
from darts import TimeSeries
from darts.models import (
    NaiveSeasonal,
    ExponentialSmoothing,
    Theta, 
    FFT,
    LightGBMModel
)
from darts.metrics import mase
from darts.utils.utils import ModelMode, SeasonalityMode

In [ ]:
# Custom 
import plotly.figure_factory as ff
from src.utils.ts_utils import rmsse

In [ ]:
np.random.seed()

In [ ]:
os.makedirs("imgs/chapter_19", exist_ok=True)

preprocessed = Path.home() / "Desktop" / "data" / "london_smart_meters" / "preprocessed"

output = Path.home() / "Desktop" / "data" / "london_smart_meters" / "output"

In [ ]:
from darts.metrics.metrics import _get_values_or_raise


def rmsse(
    actual_series,
    pred_series,
    insample,
    m=1,
    intersect=True,
    *,
    reduction=np.mean,
):

    def _multivariate_mase(
        actual_series,
        pred_series,
        insample,
        m,
        intersect,
        reduction,
    ):

        assert (
            actual_series.width == pred_series.width
        ), "The two TimeSeries instances must have the same width."

        assert (
            actual_series.width == insample.width
        ), "The insample TimeSeries must have the same width as the other series."

        assert (
            insample.end_time() + insample.freq == pred_series.start_time()
        ), "The pred_series must be the forecast of the insample series"

        insample_ = (
            insample.quantile_timeseries(quantile=0.5)
            if insample.is_stochastic
            else insample
        )

        value_list = []
        for i in range(actual_series.width):
            # old implementation of mase on univariate TimeSeries
            y_true, y_hat = _get_values_or_raise(
                actual_series.univariate_component(i),
                pred_series.univariate_component(i),
                intersect,
                remove_nan_union=False,
            )

            x_t = insample_.univariate_component(i).values()
            errors = np.square(y_true - y_hat)
            scale = np.mean(np.square(x_t[m:] - x_t[:-m]))
            assert not np.isclose(scale, 0), "cannot use MASE with periodical signals"
            value_list.append(np.sqrt(np.mean(errors / scale)))

        return reduction(value_list)

    return _multivariate_mase(
        actual_series=actual_series,
        pred_series=pred_series,
        insample=insample,
        m=m,
        intersect=intersect,
        reduction=reduction,
    )

In [ ]:
from itertools import cycle

def format_plot(
    fig,
    legends=None,
    xlabel="Time",
    ylabel="Value",
    title="",
    font_size=15,
    title_font_size=20,
    width=900,
    height=500,
):
    # Renombrar trazas si se proveen leyendas
    if legends:
        names = cycle(legends)
        fig.for_each_trace(lambda t: t.update(name=next(names)))

    fig.update_layout(
        autosize=False,
        width=width,
        height=height,
        # Título (API moderna)
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            yanchor="top",
            font=dict(size=title_font_size),
        ),
        legend_title_text=None,
        legend=dict(
            font=dict(size=font_size),
            orientation="h",
            yanchor="bottom",
            y=0.9,
            xanchor="right",
            x=1,
        ),
        # Ejes (API moderna)
        yaxis=dict(
            title=dict(text=ylabel, font=dict(size=font_size)),
            tickfont=dict(size=font_size),
        ),
        xaxis=dict(
            title=dict(text=xlabel, font=dict(size=font_size)),
            tickfont=dict(size=font_size),
        ),
        margin=dict(t=60, r=30, b=50, l=60),
        template="plotly_white",
    )
    return fig

In [ ]:
def format_y_pred(y_pred, name):
    y_pred = y_pred.data_array().to_series()
    y_pred.index = y_pred.index.get_level_values(0)
    y_pred.name = name
    return y_pred

### Reading Files

In [ ]:
# Reading the missing value imputed and train test split data
try:
    train_df = pd.read_parquet(preprocessed/"selected_blocks_train_missing_imputed.parquet")
    val_df = pd.read_parquet(preprocessed/"selected_blocks_val_missing_imputed.parquet")
    test_df = pd.read_parquet(preprocessed/"selected_blocks_test_missing_imputed.parquet")
except FileNotFoundError:
    display(HTML("""
    <div class="alert alert-block alert-warning">
    <b>Warning!</b> File not found. Please make sure you have run 01-Setting up Experiment Harness.ipynb in Chapter04
    </div>
    """))

## Generating Forecasts

In [ ]:
lcl_ids = sorted(train_df.LCLid.unique())

In [ ]:
name = "Naive"
preds = []
metrics = []

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    for lcl_id in tqdm(lcl_ids):
        model = NaiveSeasonal(K=1)
        tr = TimeSeries.from_series(train_df.loc[train_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        vl = TimeSeries.from_series(val_df.loc[val_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        model.fit(tr)
        y_pred = model.predict(len(vl))
        _mase = mase(vl, y_pred, tr)
        _rmsse = rmsse(vl, y_pred, tr)
        y_pred = format_y_pred(y_pred, "predictions").to_frame()
        y_pred['LCLid'] = lcl_id
        y_pred['Algorithm'] = name
        y_pred['energy_consumption'] = vl.data_array().to_series().values
        metric_dict = dict()
        metric_dict["LCLid"] = lcl_id
        metric_dict['Algorithm'] = name
        metric_dict['MASE'] = _mase
        metric_dict['RMSSE'] = _rmsse
        preds.append(y_pred)
        metrics.append(metric_dict)

In [ ]:
name = "SeasonalNaive"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    for lcl_id in tqdm(lcl_ids):
        model = NaiveSeasonal(K=48*7)
        tr = TimeSeries.from_series(train_df.loc[train_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        vl = TimeSeries.from_series(val_df.loc[val_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        model.fit(tr)
        y_pred = model.predict(len(vl))
        _mase = mase(vl, y_pred, tr)
        _rmsse = rmsse(vl, y_pred, tr)
        y_pred = format_y_pred(y_pred, "predictions").to_frame()
        y_pred['LCLid'] = lcl_id
        y_pred['Algorithm'] = name
        y_pred['energy_consumption'] = vl.data_array().to_series().values
        metric_dict = dict()
        metric_dict["LCLid"] = lcl_id
        metric_dict['Algorithm'] = name
        metric_dict['MASE'] = _mase
        metric_dict['RMSSE'] = _rmsse
        preds.append(y_pred)
        metrics.append(metric_dict)

In [ ]:
name = "Theta"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    for lcl_id in tqdm(lcl_ids):
        model = Theta(theta=3, seasonality_period=48*7, season_mode=SeasonalityMode.ADDITIVE)
        tr = TimeSeries.from_series(train_df.loc[train_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        vl = TimeSeries.from_series(val_df.loc[val_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        model.fit(tr)
        y_pred = model.predict(len(vl))
        _mase = mase(vl, y_pred, tr)
        _rmsse = rmsse(vl, y_pred, tr)
        y_pred = format_y_pred(y_pred, "predictions").to_frame()
        y_pred['LCLid'] = lcl_id
        y_pred['Algorithm'] = name
        y_pred['energy_consumption'] = vl.data_array().to_series().values
        metric_dict = dict()
        metric_dict["LCLid"] = lcl_id
        metric_dict['Algorithm'] = name
        metric_dict['MASE'] = _mase
        metric_dict['RMSSE'] = _rmsse
        preds.append(y_pred)
        metrics.append(metric_dict)

In [ ]:
name = "FFT"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    for lcl_id in tqdm(lcl_ids):
        model = FFT(nr_freqs_to_keep=35, trend="poly", trend_poly_degree=2)
        tr = TimeSeries.from_series(train_df.loc[train_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        vl = TimeSeries.from_series(val_df.loc[val_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        model.fit(tr)
        y_pred = model.predict(len(vl))
        _mase = mase(vl, y_pred, tr)
        _rmsse = rmsse(vl, y_pred, tr)
        y_pred = format_y_pred(y_pred, "predictions").to_frame()
        y_pred['LCLid'] = lcl_id
        y_pred['Algorithm'] = name
        y_pred['energy_consumption'] = vl.data_array().to_series().values
        metric_dict = dict()
        metric_dict["LCLid"] = lcl_id
        metric_dict['Algorithm'] = name
        metric_dict['MASE'] = _mase
        metric_dict['RMSSE'] = _rmsse
        preds.append(y_pred)
        metrics.append(metric_dict)

In [ ]:
name = "LightGBM"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    for lcl_id in tqdm(lcl_ids):
        model = LightGBMModel(lags=[-1,-2,-3,-4,-5,-6,-48,-48*7], output_chunk_length=48)
        tr = TimeSeries.from_series(train_df.loc[train_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        vl = TimeSeries.from_series(val_df.loc[val_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        model.fit(tr)
        y_pred = model.predict(len(vl))
        _mase = mase(vl, y_pred, tr)
        _rmsse = rmsse(vl, y_pred, tr)
        y_pred = format_y_pred(y_pred, "predictions").to_frame()
        y_pred['LCLid'] = lcl_id
        y_pred['Algorithm'] = name
        y_pred['energy_consumption'] = vl.data_array().to_series().values
        metric_dict = dict()
        metric_dict["LCLid"] = lcl_id
        metric_dict['Algorithm'] = name
        metric_dict['MASE'] = _mase
        metric_dict['RMSSE'] = _rmsse
        preds.append(y_pred)
        metrics.append(metric_dict)

In [ ]:
name = "ETS"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    for lcl_id in tqdm(lcl_ids):
        model = ExponentialSmoothing(trend=ModelMode.ADDITIVE, damped=True, seasonal=SeasonalityMode.ADDITIVE, seasonal_periods=48*7, random_state=42)
        tr = TimeSeries.from_series(train_df.loc[train_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        vl = TimeSeries.from_series(val_df.loc[val_df.LCLid==lcl_id, ["timestamp","energy_consumption"]].set_index("timestamp"))
        model.fit(tr)
        y_pred = model.predict(len(vl))
        _mase = mase(vl, y_pred, tr)
        _rmsse = rmsse(vl, y_pred, tr)
        y_pred = format_y_pred(y_pred, "predictions").to_frame()
        y_pred['LCLid'] = lcl_id
        y_pred['Algorithm'] = name
        y_pred['energy_consumption'] = vl.data_array().to_series().values
        metric_dict = dict()
        metric_dict["LCLid"] = lcl_id
        metric_dict['Algorithm'] = name
        metric_dict['MASE'] = _mase
        metric_dict['RMSSE'] = _rmsse
        preds.append(y_pred)
        metrics.append(metric_dict)

In [ ]:
pred_df = pd.concat(preds)
pred_df.head()

In [ ]:
pivot_df = pd.pivot_table(pred_df.reset_index(), index=["LCLid", "timestamp", "energy_consumption"], columns=['Algorithm'], values=['predictions'])
pivot_df.columns = pivot_df.columns.get_level_values(-1)
pivot_df = pivot_df.reset_index().set_index(['LCLid'])
pivot_df.head()

In [ ]:
metric_df = pd.DataFrame(metrics)
benchmark_metric_df = metric_df.loc[metric_df.Algorithm=="Naive",["LCLid",'MASE','RMSSE']].set_index("LCLid")
metric_df = metric_df.loc[metric_df.Algorithm!="Naive",["LCLid","Algorithm",'MASE','RMSSE']]
metric_df.set_index(["LCLid", "Algorithm"], inplace=True)
metric_df.head()

In [ ]:
benchmark_metric_df.head()

In [ ]:
pred_df.to_pickle(output/"preds_for_metric_analysis.pkl")
pivot_df.to_pickle(output/"pivoted_preds_for_metric_analysis.pkl")
metric_df.to_pickle(output/"metrics_for_metric_analysis.pkl")
benchmark_metric_df.to_pickle(output/"benchmark_metrics_for_metric_analysis.pkl")

### Metric Definitions

In [ ]:
def geo_mean(iterable):
    a = np.array(iterable)
    a = a[~np.isnan(a)]
    return a.prod()**(1.0/len(a))

def geo_mean_overflow(iterable):
    return np.exp(np.log(iterable).mean())

In [ ]:
def clean_inf(array):
    array[np.isinf(array)] = np.nan
    return array

In [ ]:
#MAPE
def calc_mape(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    abs_err = np.abs(a-f)
    ape = clean_inf(abs_err/a)
    return np.nanmean(ape)

#WAPE
def calc_wape(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    abs_err = np.abs(a-f)
    return np.sum(abs_err)/np.sum(np.abs(a))

#sMAPE
def calc_smape(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    abs_err = np.abs(a-f)
    ape = 2*abs_err/(a+f)
    return np.mean(ape)

#MAE
def calc_mae(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    abs_err = np.abs(a-f)
    return np.mean(abs_err)

#MdAE
def calc_mdae(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    abs_err = np.abs(a-f)
    return np.median(abs_err)

#GMAE
def calc_gmae(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    abs_err = np.abs(a-f)
    return geo_mean(abs_err)

#MSE
def calc_mse(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    sq_err = np.power(np.abs(a-f),2)
    return np.mean(sq_err)


#RMSE
def calc_rmse(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    sq_err = np.power(np.abs(a-f),2)
    return np.sqrt(np.mean(sq_err))

#MRAE
def calc_mrae(df, fc_col, ac_col, benchmark_col):
    a = df[ac_col].values
    f = df[fc_col].values
    f_star = df[benchmark_col].values
    abs_err = np.abs(a-f)
    ape = clean_inf(abs_err/np.abs((a-f_star)))
    return np.nanmean(ape)

#GMRAE
def calc_gmrae(df, fc_col, ac_col, benchmark_col):
    a = df[ac_col].values
    f = df[fc_col].values
    f_star = df[benchmark_col].values
    abs_err = np.abs(a-f)
    ape = clean_inf(abs_err/np.abs((a-f_star)))
    return geo_mean(ape)

#MdRAE
def calc_mdrae(df, fc_col, ac_col, benchmark_col):
    a = df[ac_col].values
    f = df[fc_col].values
    f_star = df[benchmark_col].values
    abs_err = np.abs(a-f)
    ape = clean_inf(abs_err/np.abs((a-f_star)))
    return np.nanmedian(ape)

# #MASE
# def calc_mase(df, fc_col, ac_col, benchmark_col):
#     a = df[ac_col].values[1:]
#     f = df[fc_col].values[1:]
#     f_star = df[ac_col].shift(1).values[1:]
#     abs_err = np.abs(a-f)
#     ape = abs_err/np.mean(np.abs(a-f_star))
#     return np.mean(ape)

# # RMSSE
# def calc_rmsse(df, fc_col, ac_col):
#     a = df[ac_col].values[1:]
#     f = df[fc_col].values[1:]
#     f_star = df[ac_col].shift(1).values[1:]
#     sq_err = np.square(a-f)
#     scale = np.mean(np.square(a-f_star))
#     return np.sqrt(np.mean(sq_err/scale))

#RelMAE
def calc_rmae(df, fc_col, ac_col, benchmark_col):
    a = df[ac_col].values
    f = df[fc_col].values
    f_star = df[benchmark_col].values
    abs_err = np.abs(a-f)
    abs_err_star = np.abs(a-f_star)
    return np.mean(abs_err)/np.mean(abs_err_star)

#RelRMSE
def calc_rrmse(df, fc_col, ac_col, benchmark_col):
    a = df[ac_col].values
    f = df[fc_col].values
    f_star = df[benchmark_col].values
    sq_err = np.square(a-f)
    sq_err_star = np.square(a-f_star)
    return np.sqrt(np.mean(sq_err))/np.sqrt(np.mean(sq_err_star))

#RelMAPE
def calc_rmape(df, fc_col, ac_col, benchmark_col):
    a = df[ac_col].values
    f = df[fc_col].values
    f_star = df[benchmark_col].values
    perc_err = clean_inf(np.abs(a-f)/a)
    perc_err_star = clean_inf(np.abs(a-f_star)/a)
    return np.nanmean(perc_err)/np.nanmean(perc_err_star)

#CFE
def calc_cfe(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    err = a-f
    return np.sum(err)

#FB
def calc_fb(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    err = a-f
    return np.sum(err)/np.sum(a)

#PB
def calc_pbmae(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    f_star = df[benchmark_col].values
    abs_err = np.abs(a-f)
    abs_err_star = np.abs(a-f_star)
    # mae_star = np.mean(abs_err_star)
    # mae = np.mean(abs_err)
    mask = abs_err<abs_err_star
    return np.sum(mask)/len(mask)

### Calculating Item-Level Metrics

In [ ]:
fns = {
    "MAPE": calc_mape,
    "WAPE": calc_wape,
    "sMAPE": calc_smape,
    "MAE": calc_mae,
    "MdAE" : calc_mdae,
    # "GMAE": calc_gmae,
    "MSE": calc_mse,
    "RMSE": calc_rmse,
    "MRAE": calc_mrae,
    # "GMRAE": calc_gmrae,
    # "RMSSE": calc_rmsse,
    "RelMAE": calc_rmae,
    "RelRMSE": calc_rrmse,
    "RelMAPE": calc_rmape,
    "CFE": calc_cfe,
    "FB" : calc_fb,
    "PB(MAE)": calc_pbmae
}

fcs = metric_df.reset_index()["Algorithm"].unique()

In [ ]:
for lcl_id in tqdm(lcl_ids):
    df = pivot_df.xs(lcl_id)
    for metric, fn in fns.items():
        for fc in fcs:
            metric_df.loc[(lcl_id,fc), metric] = fn(df, fc, "energy_consumption", "Naive")

In [ ]:
metric_df

### Calculating Aggregate Metrics

In [4]:
def calc_nd(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    abs_err = np.abs(a-f)
    return np.sum(abs_err)/np.sum(np.abs(a))

def calc_nrmse(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    sq_err = np.square(a-f)
    return np.sum(sq_err)/np.sum(np.square(a))

def calc_forecast_bias(df, fc_col, ac_col, benchmark_col=None):
    a = df[ac_col].values
    f = df[fc_col].values
    err = a-f
    return np.sum(err)/np.sum(a)

In [ ]:
agg_metrics = []

for fc in fcs:
    met_dict = {
        "Algorithm": fc,
        "meanMASE": metric_df.xs(fc, level=1)["MASE"].mean(), 
        "meanRMSSE": metric_df.xs(fc, level=1)["RMSSE"].mean(), 
        "meanWAPE":  metric_df.xs(fc, level=1)["WAPE"].mean(),
        "meanMRAE": metric_df.xs(fc, level=1)["MRAE"].mean(),
        "AvgRelRMSE": geo_mean(clean_inf(metric_df.xs(fc, level=1)["RelRMSE"].values)),
        "ND": calc_nd(pivot_df, fc_col = fc, ac_col="energy_consumption"),
        "NRMSE": calc_nrmse(pivot_df, fc_col = fc, ac_col="energy_consumption"),
        "Forecast Bias": calc_forecast_bias(pivot_df, fc_col = fc, ac_col="energy_consumption"),
    }
    agg_metrics.append(met_dict)

In [ ]:
agg_metrics_df = pd.DataFrame(agg_metrics)

agg_metrics_df

### Ranking and Plotting Aggregate Metrics

In [ ]:
# Ranking each algorithm based on different metrics
rank_df = agg_metrics_df.copy()
cols = agg_metrics_df.columns.tolist()
cols.remove("Algorithm")

for col in cols:
    rank_df[col] = rank_df[col].rank(ascending=True)
rank_df.head()

In [ ]:
cols = ["meanMASE", "meanWAPE", "ND", "meanRMSSE", "NRMSE", "meanMRAE", "Forecast Bias","AvgRelRMSE"]

In [ ]:
#Plotting an annotated heatmap
font_size = 18
corr_df = rank_df[cols].corr(method="pearson")

# to mask upper triangle
# mask = np.triu(np.ones_like(corr_df, dtype=np.bool))
# corr_df = corr_df.mask(mask)

fig = ff.create_annotated_heatmap(
    z=corr_df.to_numpy().round(2),
    x=list(corr_df.index.values),
    y=list(corr_df.columns.values),
    xgap=3,
    ygap=3,
    zmin=-1,
    zmax=1,
    colorscale="RdYlGn",
    showscale=True,
    colorbar_thickness=30,
    colorbar_ticklen=3,
)
fig.update_layout(
    # title_text="Spearman's Correlation between <br> Aggregate Rankings of Forecast Methods",
    title_x=0.5,
    titlefont={"size": 24},
    width=600,
    height=600,
    xaxis_showgrid=False,
    xaxis={"side": "bottom", "tickfont":dict(size=font_size)},
    yaxis_showgrid=False,
    yaxis_autorange="reversed",
    template="plotly_white",
    yaxis=dict(
        tickfont=dict(size=font_size),
    ))
# NaN values are not handled automatically and are displayed in the figure
# So we need to get rid of the text manually
# Also annotation color needs to be switched to black when value is less than 0,5
# for better visibility
for i in range(len(fig.layout.annotations)):
    if fig.layout.annotations[i].text == "nan":
        fig.layout.annotations[i].text = ""
    try:
        annot = float(fig.layout.annotations[i].text)
        if abs(annot) < 0.5:
            fig.layout.annotations[i].font = {"color": "#000000"}
    except ValueError:
        pass
fig.write_image(f"imgs/chapter_19/aggregate_metric_correlation.png")
fig.show()

### Ranking and Plotting Item Level Metrics

In [ ]:
# Calculating rank of households based on each metric for every algorithm
def calc_rank(rank_df):
    for col in metric_df.columns:
        rank_df[col] = rank_df[col].rank(ascending=True)

    for col in ['PB(MAE)']:
        rank_df[col] = rank_df[col].rank(ascending=False)

    for col in ['CFE', 'FB']:
        rank_df[col] = np.abs(rank_df[col]).rank(ascending=True)
    return rank_df

item_rankings =metric_df.groupby('Algorithm').apply(calc_rank)

In [6]:
cols = [
    "MASE",
    "RMSSE",
    "MAPE",
    "WAPE",
    "sMAPE",
    "MAE",
    "MdAE",
    "MSE",
    "RMSE",
    "MRAE",
    "RelMAE",
    "RelRMSE",
    "RelMAPE",
    "PB(MAE)",
    "CFE",
    "FB",
]

In [ ]:
corr_df = item_rankings[cols].corr(method='pearson')
# to mask upper triangle
# mask = np.triu(np.ones_like(corr_df, dtype=np.bool))
# corr_df = corr_df.mask(mask)
fig = px.imshow(corr_df, template="plotly_white", color_continuous_scale="RdYlGn", range_color=[-1,1])
fig.update_layout(
    # title_text="Spearman's Correlation between <br> Aggregate Rankings of Forecast Methods",
    # title_x=0.5,
    # titlefont={"size": 24},
    width=600,
    height=600,
    xaxis_showgrid=False,
    xaxis={"side": "bottom", "tickfont":dict(size=font_size)},
    yaxis_showgrid=False,
    yaxis_autorange="reversed",
    template="plotly_white",
    yaxis=dict(
        tickfont=dict(size=font_size),
    ))
fig.write_image(f"imgs/chapter_19/item_lvl_metric_correlation.png")
fig.show()